# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
%cd /content/walmart-hurricane-project

/content/walmart-hurricane-project


In [2]:
team_setup = {
    'team_name': 'HMKD',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Henna Panjshiri', 'github': 'hpanjshiri'},
        {'name': 'Maya Yonathan', 'github': 'kaavyaindrayan'},
        {'name': 'Kaavya Indrayan', 'github': 'kaavyaindrayan'},
        {'name': 'Dina Belete', 'github': 'amranbelete'},
    ],
    'repo_url': 'https://github.com/hpanjshiri/walmart-hurricane-project/tree/main',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/hpanjshiri/walmart-hurricane-project/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Henna',           # nice to have
    'meeting_time': 'Sundays 2pm, After class 11am (Mon, Wed)',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['SKU fragmentation', 'category label variants', '886 missing timestamps', 'store ids in 48 formats', '130 negative quanities.'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [3]:
!git clone https://github.com/hpanjshiri/walmart-hurricane-project.git

fatal: destination path 'walmart-hurricane-project' already exists and is not an empty directory.


In [4]:
import subprocess

REPO_PATH = '/content/walmart-hurricane-project'

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stderr)

print(result.stdout.strip() or 'no commits found for CHARTER.md')


f1e3983 Fixed CHARTER.md
2faa8b8 charter added


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [5]:
# TODO: I have all of these formatting in my separate working file with organization and separate cells. It will be attached in scratch folder
import pandas as pd
import numpy as np, re, os
import matplotlib.pyplot as plt

def find_data(filename, folders=('data', '../data', '../../data', '/content/walmart-hurricane-project/data', '/content/data', '/kaggle/input', '/content')):
  for folder in folders:
      if os.path.exists(os.path.join(folder, filename)):
          return os.path.join(folder, filename)
  raise FileNotFoundError(f'{filename} not found. Put data files in repo in the data/ folder.')

DECISIONS = [] # Storing every cleaning decision made
COUNTS = [] # Rows before then after each step

def log(step, decision, rows_affected, dropped=False):
  # dropped = True means those rows were removed and validation will use it to check the row math
  DECISIONS.append({'step': step, 'decision': decision, 'rows': int(rows_affected), 'dropped': dropped})
  print(f'[{step}] {decision} ({int(rows_affected)} row(s))')

def run_step(name, fn, df, *args):
  # Used to run a cleaning function and record the row count before then after
  before = len(df)
  out = fn(df, *args)
  COUNTS.append({'step': name, 'rows_before': before, 'rows_after': len(out)})
  return out

def show_log():
  return pd.DataFrame(DECISIONS)

def show_counts():
  return pd.DataFrame(COUNTS)

pd.set_option('display.max_colwidth', 90)
#print('pandas', pd.__version__)


# 1. Extracting the Data
def extract():
  tx = pd.read_csv(find_data('walmart_transactions.csv'))
  stores = pd.read_csv(find_data('store_locations.csv'))
  catalog = pd.read_csv(find_data('product_catalog.csv'))
  print(f'transactions: {len(tx):,} rows | stores: {len(stores)} | catalog: {len(catalog)}')
  return tx, stores, catalog

tx_raw, stores_raw, catalog_raw = extract()
#tx_raw.head()

# Raw profile to see the uncleaned version later
print(tx_raw.dtypes, '\n')
print('nulls per col:')
print(tx_raw.isnull().sum(), '\n')
print('exact dup rows:', tx_raw.duplicated().sum())
print('transaction_id repeats:', tx_raw['transaction_id'].duplicated().sum())

# Hidden missing values as strings because pandas doesn't treat str types as null
for col in tx_raw.select_dtypes(include='object').columns:
  s = tx_raw[col].dropna().astype(str).str.strip().str.lower()
  hits = s[s.isin(['', 'none', 'null', 'n/a', 'na', 'nan', '-', '--', '?', 'unknown', 'missing', 'tbd'])]
  if len(hits):
    print(f'hidden placeholders in {col}:', hits.value_counts().to_dict())


# 2. Transform
PLACEHOLDERS = {'', 'none', 'null', 'n/a', 'na', 'nan', '-', '--', '?', 'unknown', 'missing', 'tbd'}

def standardize_missing(df):
  # Turn placeholder strings into real NaN, trim spaces
  out = df.copy()
  for col in out.select_dtypes(include='object').columns:
    stripped = out[col].astype('string').str.strip()
    is_placeholder = stripped.str.lower().isin(PLACEHOLDERS) & out[col].notna()
    out[col] = stripped.where(~is_placeholder, other=pd.NA).astype(object)
    out[col] = out[col].where(out[col].notna(), np.nan)
    if is_placeholder.sum():
      log('missing_placeholders', f'{col}: {int(is_placeholder.sum())} placeholder strings converted to NaN',
        is_placeholder.sum())
  log('whitespace', 'checked all text cols for extra spaces', len(out))
  return out

def handle_required_nulls(df):
  # Dropping rows with no sku because they can't be tied to a product, other nulls kept and handled based on the step that owns that col
  out = df[df['sku'].notna()].copy()
  log('sku_missing', 'dropped rows with no sku (no tie to a product)', len(df) - len(out), dropped=True)
  for col, why in [('timestamp', 'kept; the sale is real, just cant sit on a day'),
                   ('unit_price', 'kept as NaN; we cannot make up a price. Revenue questions will skip these rows'),
                   ('store_id', 'kept as NaN; counts toward totals')]:
    if col in out.columns:
      log(f'null_{col}', f'{col} nulls {why}', out[col].isna().sum())
  return out

def drop_duplicates(df):
  out = df.drop_duplicates().copy()
  log('duplicates', 'dropped exact duplicate rows (kept first copy)', len(df) - len(out), dropped=True)
  # Reporting same id with different contents
  repeated_ids = out['transaction_id'].duplicated().sum()
  print('rows sharing a transaction_id after dedupe:', repeated_ids)
  log('duplicate_ids', 'rows sharing a transaction_id with diff contents (only reported)', repeated_ids)
  return out

def fix_prices(df):
  out = df.copy()
  cleaned = out['unit_price'].astype('string').str.replace(r'[$,\s]', '', regex=True)
  out['unit_price'] = pd.to_numeric(cleaned, errors='coerce').astype(float)
  new_nan = out['unit_price'].isna().sum() - df['unit_price'].isna().sum()
  log('prices', 'stripped $, commas, spaces and converted unit_price to float', len(out))
  log('prices_unparseable', 'values that could not become numbers', max(new_nan, 0))
  log('prices_nonpositive', 'unit_price <= 0 (only reported)', (out['unit_price'] <= 0).sum())
  return out

def fix_quantity(df):
  out = df.copy()
  out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
  # Drop quantity if none, cant add to total
  n_missing = out['quantity'].isna().sum()
  out = out[out['quantity'].notna()].copy()
  log('quantity_missing', 'dropped rows with no / unreadable quantity', n_missing, dropped=True)
  # Flag negatives but keep them
  out['quantity'] = out['quantity'].astype(int)
  out['is_return'] = out['quantity'] < 0
  log('quantity_negative', 'kept negative quantities, flagged is_return=True', out['is_return'].sum())
  log('quantity_zero', 'quantity 0 (kept, reported only)', (out['quantity'] == 0).sum())
  return out

def norm_store_id(value):
  if pd.isna(value):
    return np.nan
  key = re.sub(r'[^A-Z0-9]', '', str(value).upper())
  m = re.match(r'^([A-Z]+)0*(\d+)$', key)
  return f'{m.group(1)}-{m.group(2)}' if m else key

def fix_store_ids(df, stores):
  out = df.copy()
  before = out['store_id'].nunique()
  out['store_id'] = out['store_id'].apply(norm_store_id)
  log('store_id', f'normalized store ids to XX-### ({before} variants -> {out["store_id"].nunique()})', len(out))
  valid = set(stores['store_id'].apply(norm_store_id))
  unmatched = out[out['store_id'].notna() & ~out['store_id'].isin(valid)]['store_id']
  if len(unmatched):
    print('WARNING: store ids not in store_locations:', sorted(unmatched.unique()))
  log('store_id_unmatched', 'rows whose store_id is not in store_locations (reported & kept)', len(unmatched))
  return out

def fix_timestamps(df):
  out = df.copy()
  parsed = pd.to_datetime(out['timestamp'], format='mixed', errors='coerce')
  unparseable = (out['timestamp'].notna() & parsed.isna()).sum()
  out['timestamp'] = parsed
  out['date'] = out['timestamp'].dt.normalize()
  # Kept rows with no timestap
  log('timestamps', 'parsed mixed formats; kept rows with no timestamp (date = NaT)', out['timestamp'].isna().sum())
  log('timestamps_unparseable', 'text timestamps that would not parse (now NaT)', unparseable)
  in_range = out['date'].between('2024-09-01', '2024-09-30') | out['date'].isna()
  log('timestamps_out_of_range', 'dates outside Sept 2024 (check day/month swap)', (~in_range).sum())
  print('date range:', out['date'].min().date(), 'to', out['date'].max().date())
  return out


# 3. Validate:
def validate(clean, raw_rows):
  dropped = sum(d['rows'] for d in DECISIONS if d['dropped'])
  assert clean.duplicated().sum() == 0, 'exact duplicates survived'
  assert len(clean) == raw_rows - dropped, f'row count {len(clean)} != {raw_rows} - {dropped} from the log'
  assert clean['unit_price'].dtype == float, 'unit_price is not numeric'
  assert clean['sku'].notna().all(), 'sku has nulls'
  assert pd.api.types.is_integer_dtype(clean['quantity']), 'quantity is not an integer'
  assert clean['store_id'].dropna().str.match(r'^[A-Z]+-\d+$').all(), 'store_id has a bad format'
  assert pd.api.types.is_datetime64_any_dtype(clean['date']), 'date is not datetime'
  for col in clean.select_dtypes(include="object").columns:
    left = clean[col].dropna().astype(str).str.strip().str.lower().isin(PLACEHOLDERS).sum()
    assert left == 0, f'{col} still has placeholder strings'
  print('all validation checks passed')


# 4. Running the Pipeline:
def run_pipeline():
  DECISIONS.clear(); COUNTS.clear()
  tx, stores, catalog = extract()
  raw_rows = len(tx)

  df = run_step('standardize_missing', standardize_missing, tx)
  df = run_step('handle_required_nulls', handle_required_nulls, df)
  df = run_step('drop_duplicates', drop_duplicates, df)
  df = run_step('fix_prices', fix_prices, df)
  df = run_step('fix_quantity', fix_quantity, df)
  df = run_step('fix_store_ids', fix_store_ids, df, stores)
  df = run_step('fix_timestamps', fix_timestamps, df)

  validate(df, raw_rows)
  return df

clean = run_pipeline()
print(f'\n{len(tx_raw):,} raw rows -> {len(clean):,} rows after pipeline')
show_counts()

transactions: 25,562 rows | stores: 14 | catalog: 45
transaction_id    object
store_id          object
timestamp         object
sku               object
product_name      object
category          object
quantity           int64
unit_price        object
dtype: object 

nulls per col:
transaction_id      0
store_id            0
timestamp         886
sku                 0
product_name        0
category            0
quantity            0
unit_price          0
dtype: int64 

exact dup rows: 30
transaction_id repeats: 30
transactions: 25,562 rows | stores: 14 | catalog: 45
[whitespace] checked all text cols for extra spaces (25562 row(s))
[sku_missing] dropped rows with no sku (no tie to a product) (0 row(s))
[null_timestamp] timestamp nulls kept; the sale is real, just cant sit on a day (886 row(s))
[null_unit_price] unit_price nulls kept as NaN; we cannot make up a price. Revenue questions will skip these rows (0 row(s))
[null_store_id] store_id nulls kept as NaN; counts toward totals (0 r

,step,rows_before,rows_after
0,standardize_missing,25562,25562
1,handle_required_nulls,25562,25562
2,drop_duplicates,25562,25532
3,fix_prices,25532,25532
4,fix_quantity,25532,25532
5,fix_store_ids,25532,25532
6,fix_timestamps,25532,25532


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [6]:
# TODO: print or display your decision log
show_log()

,step,decision,rows,dropped
0,whitespace,checked all text cols for extra spaces,25562,False
1,sku_missing,dropped rows with no sku (no tie to a product),0,True
2,null_timestamp,"timestamp nulls kept; the sale is real, just cant sit on a day",886,False
3,null_unit_price,unit_price nulls kept as NaN; we cannot make up a price. Revenue questions will skip t...,0,False
4,null_store_id,store_id nulls kept as NaN; counts toward totals,0,False
5,duplicates,dropped exact duplicate rows (kept first copy),30,True
6,duplicate_ids,rows sharing a transaction_id with diff contents (only reported),0,False
7,prices,"stripped $, commas, spaces and converted unit_price to float",25532,False
8,prices_unparseable,values that could not become numbers,0,False
9,prices_nonpositive,unit_price <= 0 (only reported),0,False


(Asked claude to put show_log() results into colab table format)

| # | Problem | Rows | What I did | Why | Other option |
|---|---|---|---|---|---|
| 1 | Hidden missing values ("NULL", "N/A", blanks) | 0 | Added a step that converts placeholder text to NaN and trims spaces | pandas already turned them into NaN when loading, so the step found nothing extra. I kept it as a safety check | Fill with 0 or "Unknown", which hides the gap |
| 2 | Missing `sku` | 0 | Rule: drop the row | A sale with no product can't count toward any category | Keep as "UNKNOWN", which adds noise |
| 3 | Missing `timestamp` | 884 (3.5%) | Kept, date left blank | The sales are real, they just can't sit on a day. They're left out of daily charts and weather joins, and I'll report how many | Drop them, which loses real demand |
| 4 | Missing `unit_price` / `store_id` | 0 | Rule: keep as NaN | I won't invent a price | Fill price with a per-SKU median, but that's a guess |
| 5 | Exact duplicate rows | 30 | Dropped, kept the first copy | All 30 repeated `transaction_id`s were true copies, and none are left | Dedupe on `transaction_id` only, which is risky if ids get reused |
| 6 | Prices stored as text | 25,532 | Stripped `$`, commas and spaces, converted to float | Can't do math on strings. 0 values failed to convert | none |
| 7 | Prices <= 0 | 0 | Checked, nothing to do | | |
| 8 | Missing / unreadable `quantity` | 0 | Rule: drop the row |

### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [7]:
# TODO
#checking how many distinct skus we started with, and how much we have now
print('raw skus:', tx_raw['sku'].nunique())
print('clean skus:', clean ['sku'].nunique())

'''
We started off with 45 raw skus, and we are at 45 after cleaning. This makes sense
because we only dropped rows where skus were missing. dropping these rows
removes the blanks, but not real sku values, meaning the count wouldn't change

checked that no skus differ by case or spacing, so that the 45 does not contain
formatting varients
'''

NOISE = {'pack','pk','roll','of','the','and','led','bottled','baby','infant','phone','stick','device','roku'}
SYN   = {'batteries':'battery','candles':'candle','sz':'size'}

#grouping skus into canonical products using normalized product name
def canonical_key(name):
    s = str(name).lower()
    s = re.sub(r'pop[-\s]?tarts?', 'poptarts', s)    # pop-tarts / pop-tart / pop tarts etc
    s = re.sub(r'(\d+)\s*-?\s*(pack|pk|roll)', r'\1', s)   # 24pk / 24-pack / 12-roll -> 24 / 12 stuff
    toks = set()
    for t in re.findall(r'[a-z0-9]+', s):
        t = SYN.get(t, t)
        if t in NOISE: continue
        if t.endswith('s') and len(t) > 3 and t not in ('poptarts', 'diapers'):
            t = t[:-1]                                       # crude singularize
        toks.add(t)
    return ' '.join(sorted(toks))                            # sorted so word order doesn't matter

sku_map = catalog_raw[['sku', 'product_name']].copy()
sku_map['canonical_key'] = sku_map['product_name'].apply(canonical_key)
sku_map['canonical_id']  = sku_map.groupby('canonical_key').ngroup() + 1

groups = sku_map.groupby('canonical_key')['sku'].nunique().sort_values(ascending=False)


#the results
print('skus we started with:  ', clean['sku'].nunique())
print('canonical products now:', sku_map['canonical_id'].nunique())
print('largest group:', groups.index[0], '->', groups.iloc[0], 'SKUs')
print('transaction SKUs missing from map:', set(clean['sku']) - set(sku_map['sku']))


# pop tarts mapping
sku_map[sku_map['canonical_key'] == 'poptarts strawberry'][['sku','product_name','canonical_id']]

raw skus: 45
clean skus: 45
skus we started with:   45
canonical products now: 20
largest group: poptarts strawberry -> 4 SKUs
transaction SKUs missing from map: set()


,sku,product_name,canonical_id
0,PT-12,pop-tarts strawberry,19
1,SKU#459812,Strawberry Pop-Tart,19
2,POPTART-STRAW,Pop-Tarts Strawberry,19
3,PT_STRAWBERRY,pop-tarts strawberry,19


### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [8]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 28.54,
    "longitude": -81.38,
    "start_date": "2024-09-01",
    "end_date": "2024-09-07",
    "daily": "precipitation_sum,temperature_2m_max",
    "timezone": "America/New_York"
}

session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)
session.mount("https://", HTTPAdapter(max_retries=retry))

try:
    response = session.get(url, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()

    weather_df = pd.DataFrame(data["daily"])
    weather_df["date"] = pd.to_datetime(weather_df["time"]).dt.date
    weather_df["source"] = "open-meteo"
    weather_df = weather_df.drop(columns=["time"])

    display(weather_df.head())
    print("Rows returned:", len(weather_df))

except (requests.exceptions.RequestException, ValueError, KeyError) as e:
    print("Weather API error:", repr(e))


,precipitation_sum,temperature_2m_max,date,source
0,3.0,31.5,2024-09-01,open-meteo
1,12.4,29.6,2024-09-02,open-meteo
2,24.8,30.9,2024-09-03,open-meteo
3,22.1,31.0,2024-09-04,open-meteo
4,22.7,31.6,2024-09-05,open-meteo


Rows returned: 7


We are using the Open-Meteo Historical Weather API to collect daily precipitation and temperature data, which we plan to compare against Walmart sales during hurricane-related weather events. The API request includes three retries with exponential backoff for temporary failures. We also added a `source` column to identify where the weather data came from.

The weather observations are grouped by local date using the `America/New_York` timezone. Before merging weather with sales, we will convert the sales timestamps to the same timezone and extract the calendar date. We will then join the datasets by date and store location so that each store is matched with the appropriate weather observations.

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [9]:
# Not started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [10]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': 25532,        # an integer
    'decisions_logged': 18,           # how many entries in your log
    'canonical_products': 20,         # distinct products after consolidation
    'weather_api_rows': 0,           # rows returned by your API call
    'notebook_runs_clean': True,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25532
  done - decisions_logged = 18
  done - canonical_products = 20
  done - weather_api_rows = 0
  done - notebook_runs_clean = True

required items complete: True
progress items done: 5 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** Our biggest risk is that the cleaning pipeline can look done while still hiding mistakes, primarily in the SKU consolidation. Merging products is a judgement call, and wrong merges can change Q1 and Q2 answers without any error showing up. We are also splitting our time with other commitments, so I'm most likely to rush the review. Currently, the checks only tell us the code runs, not that our merges are right.

**Plan, with an owner and a date:** Henna will finish the remaining cleaning steps and Dina will finish the SKU consolidation by Wednesday and send the team the clean file and decision log. Kaavya and Maya will each review the cleaning code and the decision log by Thursday night. They will check each SKU merge against the product catalog and look at the review table. We will fix anything found before the Friday checkpoint begins, and we won't give the clean file to the question owners until two people besides me have signed off. We plan to be done earlier so we can practice our presentation on Sunday.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | Draft answer | Henna |
| Q2 SKU consolidation | In progress | Dina |
| Q3 weather correlation | In progress | Kaavya |
| Q4 store and geography | Not started | Dina |
| Q5 signal or artifact | Not started | Maya |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.